In [56]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt 
import pdfplumber
import torch
import torch.nn as nn
from torch.nn import functional as F 

In [13]:
text = ""
with pdfplumber.open("dataset.pdf") as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [14]:
print(len(text))

1525489


In [15]:
print(text[:1000])

DOUGLAS ADAMS
THE ULTIMATE
HITCHHIKER'S GUIDE
Complete & Unabridged
Contents:
Introduction: A Guide to the Guide
The Hitchhiker's Guide to the Galaxy
The Restaurant at the End of the Universe
Life, the Universe and Everything
So Long, and Thanks for All the Fish
Young Zaphod Plays It Safe
Mostly Harmless
FootnotesIntroduction: A GUIDE TO
THE GUIDE
Some unhelpful remarks from the author
The history of The Hitchhiker's Guide to the Galaxy is now so
complicated that every time I tell it I contradict myself, and whenever
I do get it right I'm misquoted. So the publication of this omnibus
edition seemed like a good opportunity to set the record straight (cid:884) or
at least firmly crooked. Anything that is put down wrong here is, as far
as I'm concerned, wrong for good.
The idea for the title first cropped up while I was lying drunk in a
field in Innsbruck, Austria, in 1971. Not particularly drunk, just the
sort of drunk you get when you have a couple of stiff Gössers after
not having eate

In [25]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print("".join(chars))
print(vocab_size)


 !"%&'()*,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz|~­èéö‐
86


In [28]:
stoi= {ch:i for i,ch in enumerate(chars)}
itos= {i:ch for i,ch in enumerate(chars)}
encode=lambda s: [stoi[c] for c in s]
decode=lambda l: [itos[c]for c in l]

In [29]:
print(encode("hello"))
print(decode(encode("hello")))

[60, 57, 64, 64, 67]
['h', 'e', 'l', 'l', 'o']


In [49]:
data=torch.tensor(encode(text),dtype=torch.long)
len(data)

1525489

In [50]:
n=int(0.9*(len(data)))
train_data=data[:n]
test_data=data[n:]

In [52]:
block_size=8
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
    context=x[:t+1]
    target=y[t]
    print("when context is :",context," target is :",target)

when context is : tensor([30])  target is : tensor(41)
when context is : tensor([30, 41])  target is : tensor(47)
when context is : tensor([30, 41, 47])  target is : tensor(33)
when context is : tensor([30, 41, 47, 33])  target is : tensor(38)
when context is : tensor([30, 41, 47, 33, 38])  target is : tensor(27)
when context is : tensor([30, 41, 47, 33, 38, 27])  target is : tensor(45)
when context is : tensor([30, 41, 47, 33, 38, 27, 45])  target is : tensor(1)
when context is : tensor([30, 41, 47, 33, 38, 27, 45,  1])  target is : tensor(27)


In [59]:
batch_size=4
block_size=8
def get_batch(split):
    data=train_data if split=="train" else test_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [61]:
xb,yb=get_batch("train")

In [71]:
class BigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)
    def forward(self,idx,targets=None):
        logits=self.token_embedding_table(idx)
        if targets==None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return logits,loss
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss=self(idx)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx
m=BigramLanguageModel(vocab_size)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)

torch.Size([32, 86])
tensor(4.9614, grad_fn=<NllLossBackward0>)


In [113]:
result=decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist())
for i in result:
    print(i,end="")


,3)m~X2spAfImpUq56U*èJ‐­é2­.‐R-Y!öiflxsEIePèkFAg|6S%RCDN&aAe;454v-eEZ­B1OkQUXIQ/‐W*fKYb8y./G:%R‐~qiE

In [118]:
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

In [119]:
batch_size=32
for steps in range(100000):
    xb,yb=get_batch("train")
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.3535938262939453


In [121]:
result=decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=1000)[0].tolist())
for i in result:
    print(i,end="")


Areve oke wimll atyonde atobesur
g m
" lyorveand Trex
"Olin.
Sl An Ves thad msthime a hedorothicke ay Ste poceeroincuthucounsy icour ind anum
"in Fovatong
hind fomarwh, ld Hetitthemn "I apllde wo tasut acrkicritous
Tharifoket plly fingrorinthef
"t
"Heus ched a pigear thicar-­‐ckkins d, aurgg, s. t atond."The idis. tinth ecrere and hocind l. plitalighitindenicod s.
Houg tund. " Thy t thledonloust t I trs ionar trofed Ond:84) ugllind bird, ned f favengevelkndofahelallllly waban, r g. and an
"touns ongul, coway pson hemordod It tatif memok theaineag s ngosass Helth ty or hanca omourloldicte Th hy Sheshag g adrig ts Ded panect asty,"Isllvedely anthe nd stherd nalywast ofiverut, whicefo Idaiondivan 'ted ath t mbrked. woug adouncof ey ssf ashsalson,"Be'mef s. Tim, ad te anored temathas iveclavedre ff,"Fowhubyeant.
". e s ine gy bss aithashoine mompoweente thaursspuse crthect, d suitned 30-­‐Zabawaky whis wart aply f Hid y's id apu tafe titowor inones g s inegh," pap ceret a w bofid thend pe